# RQ1 보강 D: 영역(ROI)별 하위공간, 사물 정보로 설명되는 정도(D1), 사물 정보를 뺀 맥락(D2)
코드는 `src/fmri/axes_roi.py`, `src/fmri/axes_object_check.py`, `src/fmri/axes_k_curve.py --roi`.

**먼저 Drive 갱신**: 피험자 4명의 `roi_groups.npz`를 다시 올린다(세부 영역 마스크 추가: places, faces, bodies, words, S_early, S_midventral, S_midlateral, S_midparietal, S_ventral, S_lateral, S_parietal). 로컬 `data/raw/nsd_fmri/subj0N/roi_betas/roi_groups.npz`를 `fmri_capstone/fmri/roi_betas/`(피험자 1)와 `fmri_capstone/fmri_subj0N/`(2·5·7)에 덮어쓴다.

판단 규칙(결과를 보기 전에 기록):
- **영역별**: 영역마다 k*와 방향이 다르고 영역 간 전이가 own보다 한참 낮으면 합친 하위공간은 영역들의 혼합이다. 영역 간 겹침이 반분할 수준이고 전이가 own과 비슷하면(차이 ≤0.01) 영역과 무관한 공유 구조다. 신뢰도 통과 복셀 150 미만 영역은 제외.
- **D1**: 상위 방향 점수가 COCO 80범주로 교차검증 R² 0.5 이상 설명되면 그 방향은 사물·범주 축으로 본다(분산 가중 R²로 하위공간 전체도 본다). 셔플 대조는 0 근처여야 한다.
- **D2**: 사물 정보를 뺀 맥락 임베딩(`ctxres_P1`)의 full P가 0.05 이상이고 셔플(≈0) 위이면 사물 이름 너머 정보가 있다고 본다. 그 위에서 RRR이 소수 k로 포화하고 피험자 간에 공유되면 사물 너머의 저차원 하위공간이 있다고 본다.

In [ ]:
from google.colab import drive
import os, shutil, time, glob

DRIVE = '/content/drive/MyDrive/fmri_capstone'
LOCAL = '/content/local'          # 입력(fMRI, 임베딩, 번들)은 로컬 복사본을 읽는다(끊김 방지)
os.makedirs(LOCAL, exist_ok=True)

def mount():
    drive.mount('/content/drive', force_remount=True)
    assert os.path.exists(f'{DRIVE}/embeddings_v2/ctx_ids.json'), 'Drive 경로를 확인하세요'

def copy_dir(src, dst, only=None):
    """Drive가 끊기면 다시 마운트하고 이어서 복사한다(이미 복사한 파일은 건너뜀)."""
    if not os.path.exists(src):
        print('없음(건너뜀):', src); return
    os.makedirs(dst, exist_ok=True)
    for f in sorted(os.listdir(src)):
        if only and not any(f.startswith(p) or f == p for p in only): continue
        s, d = f'{src}/{f}', f'{dst}/{f}'
        if os.path.isdir(s) or os.path.exists(d): continue
        for attempt in range(4):
            try:
                shutil.copy2(s, d); break
            except OSError as e:
                print('복사 재시도', f, e); time.sleep(3); mount()

mount()
!rm -rf /content/repo && git clone -q https://github.com/geon-2/fmri-imagery-context /content/repo
!pip -q install scipy
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 axes_k_curve.py --selftest | tail -1 && PYTHONPATH=/content/repo/src/labeling python3 axes_shared.py --selftest | tail -1

copy_dir(f'{DRIVE}/embeddings_v2', f'{LOCAL}/embeddings_v2')
copy_dir(f'{DRIVE}/colab_bundle', f'{LOCAL}/colab_bundle')
SRC = {'subj01': f'{DRIVE}/fmri/roi_betas', 'subj02': f'{DRIVE}/fmri_subj02', 'subj05': f'{DRIVE}/fmri_subj05', 'subj07': f'{DRIVE}/fmri_subj07'}
FM = {}
for k, v in SRC.items():
    copy_dir(v, f'{LOCAL}/fmri_{k}', only=['session', 'voxels.npz', 'roi_groups.npz', 'trial_map.csv'])
    FM[k] = f'{LOCAL}/fmri_{k}'
    print(k, len(glob.glob(FM[k] + '/session*.npy')), '세션 복사됨')

BASE = DRIVE                      # 결과(--out-dir)와 그림은 Drive에 바로 저장된다
EMB, BUNDLE = f'{LOCAL}/embeddings_v2', f'{LOCAL}/colab_bundle'   # 입력은 로컬
EXE = 'cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u'

def push_ctxres():
    """D2가 만든 ctxres 임베딩을 Drive에도 저장한다(D2 셀 직후 한 번)."""
    for f in glob.glob(f'{EMB}/*ctxres*'):
        shutil.copy2(f, f'{DRIVE}/embeddings_v2/')
    print('ctxres 파일을 Drive에 저장함')


## D1, D2 (뇌 자료는 방향을 구할 때만 쓴다). D2는 `ctxres_P1.npy`를 Drive의 embeddings_v2에 저장한다
피험자 1, MPNet과 CLIP. 몇 분.

In [ ]:
for MODEL in ['MPNet', 'CLIP']:
    print('=====', MODEL)
    !{EXE} axes_object_check.py --fmri-dir {FM['subj01']} --emb-dir {EMB} --bundle-dir {BUNDLE} --model {MODEL} --out-dir {BASE}/axes_object --tag subj01 --k 12
push_ctxres()   # ctxres_P1.npy를 Drive에도 저장

## D2 이어서: 사물 정보를 뺀 맥락(ctxres_P1)의 k 곡선 (4명, MPNet; 피험자 1은 CLIP도)
맥락 곡선(ctx_P1)과 비교한다. 부트스트랩 200회.

In [ ]:
for SUBJ, MODELS in [('subj01', ['MPNet', 'CLIP']), ('subj02', ['MPNet']), ('subj05', ['MPNet']), ('subj07', ['MPNet'])]:
    for MODEL in MODELS:
        print('=====', SUBJ, MODEL, 'ctxres_P1')
        !{EXE} axes_k_curve.py --fmri-dir {FM[SUBJ]} --emb-dir {EMB} --model {MODEL} --rep ctxres_P1 --out-dir {BASE}/axes --tag {SUBJ} --boot 200

## 영역별 분석 (4명, MPNet ctx_P1; 피험자 1은 CLIP도, 그리고 피험자 1은 ctxres_P1도)
영역을 합치지 않고 각각 적합. 영역 수 13개라 피험자당 30분~1시간 걸릴 수 있다.

In [ ]:
for SUBJ, MODELS in [('subj01', ['MPNet', 'CLIP']), ('subj02', ['MPNet']), ('subj05', ['MPNet']), ('subj07', ['MPNet'])]:
    for MODEL in MODELS:
        print('=====', SUBJ, MODEL, 'ctx_P1')
        !{EXE} axes_roi.py --fmri-dir {FM[SUBJ]} --emb-dir {EMB} --model {MODEL} --rep ctx_P1 --out-dir {BASE}/axes_roi --tag {SUBJ}

In [ ]:
# 사물을 뺀 맥락의 영역별 결과(피험자 1)
!{EXE} axes_roi.py --fmri-dir {FM['subj01']} --emb-dir {EMB} --model MPNet --rep ctxres_P1 --out-dir {BASE}/axes_roi --tag subj01
# 같은 분석을 사물 이름(obj_P1)과 원본(orig_P1)에도 (영역별 차이가 맥락 특이적인지)
for REP in ['obj_P1', 'orig_P1']:
    !{EXE} axes_roi.py --fmri-dir {FM['subj01']} --emb-dir {EMB} --model MPNet --rep {REP} --out-dir {BASE}/axes_roi --tag subj01

### 영역별 그림: 영역 × 표현의 k 곡선과 영역 간 겹침 히트맵

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
R = json.load(open(f'{BASE}/axes_roi/axes_roi_subj01_MPNet_ctx_P1.json'))
for roi, v in R['rois'].items():
    c = v['curves']['RRR']; axes[0].plot([x['k'] for x in c], [x['P'] / v['full_P'] for x in c], 'o-', label=f"{roi} (full {v['full_P']:.2f}, k*={v['kstar']['RRR']})")
axes[0].set_xscale('log'); axes[0].set_xlabel('k'); axes[0].set_ylabel('RRR P / full P (relative to each ROI)'); axes[0].legend(fontsize=6); axes[0].set_title('subj01 MPNet ctx_P1: RRR by ROI')
names = list(R['rois']); M = np.eye(len(names))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i < j: M[i, j] = M[j, i] = R['overlap']['12'].get(f'{a}|{b}', np.nan)
im = axes[1].imshow(M, vmin=0, vmax=0.8); axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=90, fontsize=7); axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels(names, fontsize=7)
plt.colorbar(im, ax=axes[1]); axes[1].set_title('subspace overlap between ROIs (k=12)')
T = np.full((len(names), len(names)), np.nan)
for b, tb in R['transfer'].items():
    r = tb['12']
    for a in names:
        if f'from_{a}' in r: T[names.index(b), names.index(a)] = r[f'from_{a}']['P'] - r['own_RRR']['P']
im = axes[2].imshow(T, vmin=-0.1, vmax=0.0, cmap='viridis'); axes[2].set_xticks(range(len(names))); axes[2].set_xticklabels(names, rotation=90, fontsize=7); axes[2].set_yticks(range(len(names))); axes[2].set_yticklabels(names, fontsize=7)
axes[2].set_xlabel('source ROI directions'); axes[2].set_ylabel('target ROI'); plt.colorbar(im, ax=axes[2]); axes[2].set_title('transfer loss: from-ROI minus own (k=12)')
plt.tight_layout(); plt.savefig(f'{BASE}/axes_roi/roi_overview_subj01_MPNet.png', dpi=150); plt.show()